In [1]:
import pyabf
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 
import os 
import csv

#directory = "Ephys/EKODE_hDRG"

current_directory = os.getcwd()
print (current_directory)

#Load Abf files
#directory = "EKODE_hDRG"
abf_files = [f for f in os.listdir(current_directory) if f.endswith(".abf")]
print (abf_files)

results = pd.DataFrame(columns=["File", "RMP", "SS", "deltaSS", "Sag", "R_in", "rebound", "rebound latency", "rebound current", "rheobase", "FSL"])

x_range1 = (0, 0.025) #RMP range
x_range2 = (0.425, 0.525) #Steady State Voltage range
x_range3 = (0.03, 0.3) #Sag voltage range
x_range4 = (0.525, 1.025) #Post current injection, recovery range 
x_range5 = (0.025, 0.525) #Current injection range

C:\Users\ZUR00A\Ephys\automated\hDRG
['HS16_01_0000.abf', 'HS16_04_0000.abf', 'HS17_03_0000.abf', 'HS17_04_0000.abf', 'HS17_05_0000.abf', 'HS17_06_0000.abf', 'HS17_07_0000.abf', 'HS17_19_0000.abf', 'HS17_21_0000.abf', 'HS17_31_0000.abf', 'HS17_32_0000.abf', 'HS17_37_0000.abf', 'HS17_38_0000.abf', 'HS17_39_0000.abf', 'HS17_54_0000.abf', 'HS17_55_0000.abf', 'HS17_56_0000.abf', 'HS17_57_0000.abf', 'HS17_82_0000.abf', 'HS17_83_0000.abf', 'HS17_84_0000.abf', 'HS17_85_0000.abf', 'HS17_86_0000.abf', 'HS22_000_0000.abf', 'HS22_001_0000.abf', 'HS22_003_0000.abf', 'HS22_004_0000.abf', 'HS22_005_0000.abf', 'HS22_006_0000.abf', 'HS22_007_0000.abf', 'HS22_020_0000.abf', 'HS22_021_0000.abf', 'HS22_027_0000.abf', 'HS22_028_0000.abf']


In [2]:
for abf_file in abf_files: #Loop through abf files
    abf_file_path = os.path.join(current_directory, abf_file)

    abf=pyabf.ABF(abf_file_path) #Open the abf file

    #set to first Sweep for getting RMP, R_in, and Sag Perecntage. 
    sweep_number = 0
    abf.setSweep(sweep_number)

    #calculate RMP from the first sweep
    x_min1, x_max1 = x_range1
    rmp = abf.sweepY[(abf.sweepX >= x_min1) & (abf.sweepX <= x_max1)].mean()

    #calculate steady state voltage (used for caluculating sag and R_in)
    x_min2, x_max2 = x_range2
    ss = abf.sweepY[(abf.sweepX >= x_min2) & (abf.sweepX <= x_max2)].mean()

    #calculate minimun value in the first 300ms of current injection (used to calculate Sag)
    x_min3, x_max3 = x_range3
    minval = abf.sweepY[(abf.sweepX >= x_min3) & (abf.sweepX <= x_max3)].min()

    #calucate differnce in steady state and RMP (used for R_in and Sag)
    delta_ss = ss-rmp

    #calculate Sag Percentage
    sag = ((minval-ss)/(minval-rmp))*100

    #calculate current injection during first sweep, used for R_in (generally -100pA) 
    time_point = 0.4

    data_index = int(time_point * abf.dataRate)
    
    I = abf.sweepC[data_index]

    #calculate R_in
    R_in = delta_ss / I *1000

    # Define the range of sweeps you want to check (first 10 sweeps for Rebound)
    start_sweep = 0
    end_sweep = 9

    # Initialize variables to store Rebound analysis results
    spike_detected = 0
    rebound_latency = np.nan
    rebound_current = np.nan
    
    # Parameters
    spike_threshold_rebound = 5  # mV
    spike_window_ms = 10  # window after threshold crossing to search for spike peak

    # Iterate through the specified sweeps for Rebound analysis
    for sweep_number in range(start_sweep, end_sweep + 1):
        # Set the sweep number
        abf.setSweep(sweepNumber=sweep_number)

        # Extract the voltage values for the last 0.5 seconds
        voltage_range4 = abf.sweepY[(abf.sweepX >= x_range4[0]) & (abf.sweepX <= x_range4[1])]
        time_range4 = abf.sweepX[(abf.sweepX >= x_range4[0]) & (abf.sweepX <= x_range4[1])]

        #Find if there is a rebound spike
        rebound_indices = np.where(voltage_range4 > spike_threshold_rebound)[0]

        # Check for a spike (for simplicity, checking if the maximum voltage is above a certain threshold)
        if rebound_indices.size > 0:
            spike_detected = 1

            #set rebound_current time point
            rebound_current_time_point = 0.3
            rebound_current = abf.sweepC[int(rebound_current_time_point * abf.dataRate)]

            # First spike index
            rebound_first_spike_index = rebound_indices[0]

            # Define window to search for peak
            window_samples = int((spike_window_ms / 1000) * abf.dataRate)
            rebound_end_index = min(rebound_first_spike_index + window_samples, len(voltage_range4))
            

            # Check if the segment is non-empty
            if rebound_end_index > rebound_first_spike_index:
                rebound_spike_segment = voltage_range4[rebound_first_spike_index:rebound_end_index]
                rebound_peak_index_in_segment = np.argmax(rebound_spike_segment)
                rebound_peak_index = rebound_first_spike_index + rebound_peak_index_in_segment

            # Calculate rebound latency (first spike after hyperpolarizing pulse)
            rebound_latency = (time_range4[rebound_peak_index] - x_range4[0]) * 1000
    
            break  # Exit the loop if a spike is detected

    # Initialize variables for Rheobase calculation
    rheobase = np.nan  # Default to NaN if no Rheobase is found
    fsl = np.nan  # Default to NaN if no spike is detected

    # Check for a spike (checking if any voltage value exceeds the threshold)
    spike_threshold_rheo = 10  # Set spike threshold for rheobase to 10 mV
    spike_window_ms = 10  # window after threshold crossing to search for spike peak

    # Set the starting sweep number to 11
    start_sweep = 11

    # Iterate through sweeps to find Rheobase
    for sweep_number in range(start_sweep, abf.sweepCount):
        abf.setSweep(sweep_number)

        # Extract the voltage values for x_range5
        voltage_range5 = abf.sweepY[(abf.sweepX >= x_range5[0]) & (abf.sweepX <= x_range5[1])]
        time_range5 = abf.sweepX[(abf.sweepX >= x_range5[0]) & (abf.sweepX <= x_range5[1])]
        spike_indices = np.where(voltage_range5 > spike_threshold_rheo)[0]
        
        if np.any(voltage_range5 > spike_threshold_rheo):
            
            # Set Rheobase as the current at x_range5
            rheobase_time_point = 0.3  # Adjust as needed
            rheobase = abf.sweepC[int(rheobase_time_point * abf.dataRate)]
            first_spike_index = spike_indices[0]

            
            # Define window to search for peak (e.g., 5 ms after threshold crossing)
            window_samples = int((spike_window_ms / 1000) * abf.dataRate)
            end_index = min(first_spike_index + window_samples, len(voltage_range5))

            
            # Find peak within window
            spike_segment = voltage_range5[first_spike_index:end_index]
            peak_index_in_segment = np.argmax(spike_segment)
            peak_index = first_spike_index + peak_index_in_segment

            
            # Calculate FSL relative to x_range5[0]
            fsl = (time_range5[peak_index] - x_range5[0]) * 1000  # in ms

            break  # Exit the loop if Rheobase is found
            
    #Save results into your dataframe        
    sweep_result = pd.DataFrame({
        "File": abf_file, 
        "RMP": rmp, 
        "SS": ss, 
        "deltaSS": delta_ss, 
        "Sag": sag, 
        "R_in": R_in,
        "rebound": spike_detected,
        "rebound latency": rebound_latency, 
        "rebound current": rebound_current,
        "rheobase": rheobase,
        "FSL": fsl}, 
            index=[0]
                          )
    results = pd.concat([results, sweep_result], axis=0, ignore_index=True)

print (results)

C:\Users\ZUR00A\AppData\Local\Temp\ipykernel_30992\812587927.py:149: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, sweep_result], axis=0, ignore_index=True)


                 File        RMP          SS    deltaSS        Sag  \
0    HS16_01_0000.abf -43.884762 -133.044357 -89.159592  13.791817   
1    HS16_04_0000.abf -44.208664 -119.497879 -75.289215  13.172992   
2    HS17_03_0000.abf -57.470558  -61.423866  -3.953308  27.551141   
3    HS17_04_0000.abf -62.255253  -79.716393 -17.461140   5.273543   
4    HS17_05_0000.abf -40.128429  -43.810314  -3.681885  31.477886   
5    HS17_06_0000.abf -54.552174  -82.925865 -28.373692  38.404137   
6    HS17_07_0000.abf -35.576931  -41.974140  -6.397209  -0.195313   
7    HS17_19_0000.abf -63.228653  -87.948364 -24.719711  59.843040   
8    HS17_21_0000.abf -67.236427  -87.806664 -20.570236  44.422221   
9    HS17_31_0000.abf -67.852371  -82.244087 -14.391716  26.728114   
10   HS17_32_0000.abf -62.523346  -76.497818 -13.974472  42.920029   
11   HS17_37_0000.abf -50.496376  -71.761154 -21.264778   9.309810   
12   HS17_38_0000.abf -54.983925  -55.990150  -1.006226  43.431398   
13   HS17_39_0000.ab

In [3]:
# Save the DataFrame to a CSV file
output_csv = "ExciteNess_hDRG_auto.csv"
results.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")

Results saved to ExciteNess_hDRG_auto.csv
